# Logistic regression

In [ ]:
!pip install -q transformers sentence-transformers accelerate bitsandbytes

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,roc_curve
)
from scipy.stats import pearsonr
from huggingface_hub import login
import random

# 使用 HuggingFace Token
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX" 
login(HF_TOKEN)

# 硬體設定
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 隨機變數
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# BATCH_SIZE
BATCH_SIZE = 25 # Hallurag

In [ ]:
def load_multi_json(path):
    decoder = json.JSONDecoder()
    objs = []

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    idx = 0
    n = len(text)

    while idx < n:
        # skip whitespace
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break

        obj, end = decoder.raw_decode(text, idx)
        objs.append(obj)
        idx = end

    return objs

In [ ]:
# ====================================================
# 1. 建構 DataFrame：從 JSON → span-level 資料表
# ====================================================
def construct_dataframe(result_file):
    print(f"[Info] 讀取結果檔案: {result_file}")
    if not os.path.exists(result_file):
        raise FileNotFoundError(f"找不到檔案: {result_file}")

    response_data = []
    response_data = load_multi_json(result_file)
    
    if not response_data:
        raise ValueError("結果檔案中沒有任何有效資料")

    # 依第一筆 span 推出 ECS / PKS 的 key
    first_score = response_data[0]["scores"][0]
    ecs_keys = list(first_score["prompt_attention_score"].keys())
    pks_keys = list(first_score["parameter_knowledge_scores"].keys())
    print(f"[Info] 偵測到 ECS heads 數量 = {len(ecs_keys)}, PKS layers 數量 = {len(pks_keys)}")

    # 準備 DataFrame 欄位
    data = {
        "identifier": [],           # resp_i_span_j
        "hallucination_label": [],  # 0 / 1
        "split": []                 # train / test
    }
    for k in ecs_keys:
        data[f"ecs_{k}"] = []
    for k in pks_keys:
        data[f"pks_{k}"] = []

    for resp_idx, item in enumerate(tqdm(response_data, desc="建構 DataFrame")):
        split = item.get("split", "train")
        for span_idx, score_item in enumerate(item["scores"]):
            identifier = f"resp_{resp_idx}_span_{span_idx}"
            data["identifier"].append(identifier)
            data["hallucination_label"].append(score_item["hallucination_label"])
            data["split"].append(split)

            for k in ecs_keys:
                data[f"ecs_{k}"].append(score_item["prompt_attention_score"].get(k, 0.0))
            for k in pks_keys:
                data[f"pks_{k}"].append(score_item["parameter_knowledge_scores"].get(k, 0.0))

    df = pd.DataFrame(data)

    print("\n[Info] 標籤分布 (0: 非幻覺, 1: 幻覺)")
    print(df["hallucination_label"].value_counts(normalize=True))

    return df, ecs_keys, pks_keys


In [ ]:
# ====================================================
# 2. 檢查 response leakage
# ====================================================
def check_response_leakage(df):
    df = df.copy()
    df["response_group"] = df["identifier"].apply(lambda x: x.split("_span_")[0])

    grp = df.groupby("response_group")["split"].nunique()
    leaked = grp[grp > 1]

    if len(leaked) > 0:
        print("發現 Response Leakage！以下是前幾個例子：")
        print(leaked.head())
        raise RuntimeError(f"共有 {len(leaked)} 個 response 同時存在於 train / test")
    else:
        print("未發現 Response Leakage（train / test 切分安全）")

In [ ]:
# ====================================================
# 3. 計算單一 feature 的 AUC（用來 ranking 頭 / layer）
# ====================================================
def calculate_feature_metrics(df, feature_keys, prefix, label_col="hallucination_label"):
    metrics = []
    y = df[label_col].values
    for key in feature_keys:
        col_name = f"{prefix}_{key}"
        x = df[col_name].values
        try:
            auc = roc_auc_score(y, x)
        except Exception:
            auc = 0.5  
        metrics.append({"key": key, "auc": auc})
        
    metrics_sorted = sorted(metrics, key=lambda d: d["auc"], reverse=True)
    return metrics_sorted

In [ ]:
# # ====================================================
# # 4. ReDeEP Regression（學 alpha / beta + 評估）
# # ====================================================

def evaluate_redeep_regression(
    train_df,
    test_df,
    top_ecs_keys,
    top_pks_keys,
    learn_threshold=True,   # 是否學 threshold
):

    train_df = train_df.copy()
    test_df = test_df.copy()

    # ====================================================
    # 1. Feature construction
    # ====================================================
    ecs_cols = [f"ecs_{k}" for k in top_ecs_keys]
    pks_cols = [f"pks_{k}" for k in top_pks_keys]

    train_df["pks_sum"] = train_df[pks_cols].sum(axis=1)
    train_df["ecs_sum"] = train_df[ecs_cols].sum(axis=1)
    test_df["pks_sum"] = test_df[pks_cols].sum(axis=1)
    test_df["ecs_sum"] = test_df[ecs_cols].sum(axis=1)

    X_train = train_df[["pks_sum", "ecs_sum"]].values
    X_test = test_df[["pks_sum", "ecs_sum"]].values
    y_train = train_df["hallucination_label"].values
    y_test = test_df["hallucination_label"].values

    # ====================================================
    # 2. Normalization (fit on train only)
    # ====================================================
    scaler = MinMaxScaler()
    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)

    # ====================================================
    # 3. Logistic Regression (learn alpha / beta)
    # ====================================================
    model = LogisticRegression(solver="liblinear", random_state=RANDOM_SEED)
    model.fit(X_train_norm, y_train)

    # ====================================================
    # 4. Scores
    # ====================================================
    train_score = model.decision_function(X_train_norm)
    test_score = model.decision_function(X_test_norm)

    # ====================================================
    # 5. Threshold learning (TRAIN ONLY)
    # ====================================================
    if learn_threshold:
        fpr, tpr, thresholds = roc_curve(y_train, train_score)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        threshold = thresholds[best_idx]
    else:
        threshold = 0.0

    # ====================================================
    # 6. Chunk-level metrics 
    # ====================================================
    y_pred_chunk = (test_score > threshold).astype(int)

    auc_chunk = roc_auc_score(y_test, test_score)
    pcc_chunk, _ = pearsonr(test_score, y_test)

    acc_chunk = accuracy_score(y_test, y_pred_chunk)
    prec_chunk = precision_score(y_test, y_pred_chunk, zero_division=0)
    rec_chunk = recall_score(y_test, y_pred_chunk, zero_division=0)
    f1_chunk = f1_score(y_test, y_pred_chunk, zero_division=0)

    # ====================================================
    # 7. Response-level metrics
    # ====================================================
    test_df["redeep_score"] = test_score
    test_df["pred_label"] = y_pred_chunk
    test_df["response_group"] = test_df["identifier"].apply(
        lambda x: x.split("_span_")[0]
    )

    grouped = test_df.groupby("response_group").agg({
        "redeep_score": "mean",
        "hallucination_label": "max",
        "pred_label": "max"
    })

    auc_resp = roc_auc_score(grouped["hallucination_label"], grouped["redeep_score"])
    pcc_resp, _ = pearsonr(grouped["redeep_score"], grouped["hallucination_label"])

    acc_resp = accuracy_score(grouped["hallucination_label"], grouped["pred_label"])
    prec_resp = precision_score(grouped["hallucination_label"], grouped["pred_label"], zero_division=0)
    rec_resp = recall_score(grouped["hallucination_label"], grouped["pred_label"], zero_division=0)
    f1_resp = f1_score(grouped["hallucination_label"], grouped["pred_label"], zero_division=0)

    return {
        # chunk-level
        "auc_chunk": auc_chunk,
        "pcc_chunk": pcc_chunk,
        "acc_chunk": acc_chunk,
        "precision_chunk": prec_chunk,
        "recall_chunk": rec_chunk,
        "f1_chunk": f1_chunk,

        # response-level
        "auc_resp": auc_resp,
        "pcc_resp": pcc_resp,
        "acc_resp": acc_resp,
        "precision_resp": prec_resp,
        "recall_resp": rec_resp,
        "f1_resp": f1_resp,

        # model params
        "alpha_learned": model.coef_[0, 0],
        "beta_learned": -model.coef_[0, 1],
        "threshold": float(threshold),  
    }

In [ ]:
# ====================================================
# 6. Main：整體流程
# ====================================================
if __name__ == "__main__":

    MODEL_KEY = "Llama-3.2-1B"
    
    # ===================== 基本設定 =====================
    INPUT_RESULT_FILE = "Llama-3.2-1B_hallurag_results.json"   # JSON 輸入 根據 stage 2 的輸出改名字
    SAVE_PATH = f"redeep_eval_result_{MODEL_KEY}.json"                      # 儲存最終結果 (JSON)，如果有的
        
    # ---- 1. 建立 DataFrame ----
    df, ecs_keys, pks_keys = construct_dataframe(INPUT_RESULT_FILE)

    # ---- 2. 檢查 response leakage ----
    check_response_leakage(df)

    # ---- 3. 使用 dataset 內建的 split ----
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)
    print(f"\n[Info] Train spans: {len(train_df)}, Test spans: {len(test_df)}")

    # ---- 4. 計算每個 ECS head / PKS layer 的單獨 AUC（用來 ranking）----
    ecs_metrics = calculate_feature_metrics(train_df, ecs_keys, prefix="ecs")
    pks_metrics = calculate_feature_metrics(train_df, pks_keys, prefix="pks")

    ecs_ranked = [m["key"] for m in ecs_metrics] 
    pks_ranked = [m["key"] for m in pks_metrics]

    print("\n[Info] Top 5 ECS heads (by AUC):", ecs_ranked[:5])
    print("[Info] Top 5 PKS layers (by AUC):", pks_ranked[:5])

    # ---- 5. Grid Search：尋找最佳 Top-N head / Top-K layer ----
    best_res = None
    best_params = None

    if MODEL_KEY == "Llama-3.2-1B":
        cand_ecs = list(range(1, 17))   # [1, 2, ..., 16]
        cand_pks = list(range(1, 17))   # [1, 2, ..., 16]
    elif MODEL_KEY == "Llama-3.2-3B":
        cand_ecs = list(range(1, 29))   # [1, 2, ..., 29]
        cand_pks = list(range(1, 29))   # [1, 2, ..., 29]


    print("\n[Info] 開始 Grid Search (Top-N heads x Top-K layers)...")
    for n_ecs in cand_ecs:
        for n_pks in cand_pks:
            n_ecs_eff = min(n_ecs, len(ecs_ranked))
            n_pks_eff = min(n_pks, len(pks_ranked))

            sel_ecs_keys = ecs_ranked[:n_ecs_eff]
            sel_pks_keys = pks_ranked[:n_pks_eff]

            res = evaluate_redeep_regression(
                train_df=train_df,
                test_df=test_df,
                top_ecs_keys=sel_ecs_keys,
                top_pks_keys=sel_pks_keys
            )

            if (best_res is None) or (res["auc_resp"] > best_res["auc_resp"]):
                best_res = res
                best_params = {
                    "top_ecs": n_ecs_eff,
                    "top_pks": n_pks_eff,
                    "ecs_keys": list(sel_ecs_keys),
                    "pks_keys": list(sel_pks_keys),
                }

    # ---- 6. 印出最終結果 ----
    print("\n================ 最終結果 (ReDeEP Regression) ================")
    print(f"最佳 Top-ECS-head 數量: {best_params['top_ecs']} → {best_params['ecs_keys']}")
    print(f"最佳 Top-PKS-layer 數量: {best_params['top_pks']} → {best_params['pks_keys']}")
    print(f"學到的 PKS 係數 (alpha): {best_res['alpha_learned']:.4f}")
    print(f"學到的 ECS 係數 (beta):  {best_res['beta_learned']:.4f}")
    print(f"學到的 threshold:  {best_res['threshold']:.4f}")

    print("\n[Chunk-level]")
    print(f"  AUC:      {best_res['auc_chunk']:.4f}")
    print(f"  PCC:      {best_res['pcc_chunk']:.4f}")
    print(f"  Accuracy: {best_res['acc_chunk']:.4f}")
    print(f"  Precision:{best_res['precision_chunk']:.4f}")
    print(f"  Recall:   {best_res['recall_chunk']:.4f}")
    print(f"  F1:       {best_res['f1_chunk']:.4f}")

    print("\n[Response-level]")
    print(f"  AUC:      {best_res['auc_resp']:.4f}")
    print(f"  PCC:      {best_res['pcc_resp']:.4f}")
    print(f"  Accuracy: {best_res['acc_resp']:.4f}")
    print(f"  Precision:{best_res['precision_resp']:.4f}")
    print(f"  Recall:   {best_res['recall_resp']:.4f}")
    print(f"  F1:       {best_res['f1_resp']:.4f}")

    print(f"\n模型訓練已完成")


# NNLS

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.optimize import nnls
from scipy.stats import pearsonr

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# ====================================================
# 0. 建構 DataFrame：從 JSON → span-level 資料表
# ====================================================
def construct_dataframe(result_file):
    print(f"[Info] 讀取結果檔案: {result_file}")
    if not os.path.exists(result_file):
        raise FileNotFoundError(f"找不到檔案: {result_file}")

    response_data = []
    response_data = load_multi_json(result_file)
    
    if not response_data:
        raise ValueError("結果檔案中沒有任何有效資料")

    # 依第一筆 span 推出 ECS / PKS 的 key
    first_score = response_data[0]["scores"][0]
    ecs_keys = list(first_score["prompt_attention_score"].keys())
    pks_keys = list(first_score["parameter_knowledge_scores"].keys())
    print(f"[Info] 偵測到 ECS heads 數量 = {len(ecs_keys)}, PKS layers 數量 = {len(pks_keys)}")

    # 準備 DataFrame 欄位
    data = {
        "identifier": [],           # resp_i_span_j
        "hallucination_label": [],  # 0 / 1
        "split": []                 # train / test
    }
    for k in ecs_keys:
        data[f"ecs_{k}"] = []
    for k in pks_keys:
        data[f"pks_{k}"] = []

    for resp_idx, item in enumerate(tqdm(response_data, desc="建構 DataFrame")):
        split = item.get("split", "train")
        for span_idx, score_item in enumerate(item["scores"]):
            identifier = f"resp_{resp_idx}_span_{span_idx}"
            data["identifier"].append(identifier)
            data["hallucination_label"].append(score_item["hallucination_label"])
            data["split"].append(split)

            for k in ecs_keys:
                data[f"ecs_{k}"].append(score_item["prompt_attention_score"].get(k, 0.0))
            for k in pks_keys:
                data[f"pks_{k}"].append(score_item["parameter_knowledge_scores"].get(k, 0.0))

    df = pd.DataFrame(data)

    print("\n[Info] 標籤分布 (0: 非幻覺, 1: 幻覺)")
    print(df["hallucination_label"].value_counts(normalize=True))

    return df, ecs_keys, pks_keys


In [ ]:
# ====================================================
# 1. Feature ranking（算單一 feature 的 AUC）
# ====================================================

def calculate_feature_metrics(df, feature_keys, prefix, label_col="hallucination_label"):
    metrics = []
    y = df[label_col].values

    for key in feature_keys:
        col = f"{prefix}_{key}"
        x = df[col].values
        try:
            auc = roc_auc_score(y, x)
        except Exception:
            auc = 0.5
        metrics.append({"key": key, "auc": auc})

    return metrics

In [ ]:
# ====================================================
# 3. 使用的到的 function
# ====================================================

def safe_pearsonr(x, y):
    try:
        return pearsonr(x, y)[0]
    except Exception:
        return None

# ====================================================
# ReDeEP NNLS
# ====================================================

def train_redeep_nnls(train_df, pks_keys, ecs_keys):
    pks_cols = [f"pks_{k}" for k in pks_keys]
    ecs_cols = [f"ecs_{k}" for k in ecs_keys]
    feature_cols = pks_cols + ecs_cols

    X_raw = train_df[feature_cols].values.astype(float)
    y = train_df["hallucination_label"].values.astype(float)

    scaler = MinMaxScaler()
    X = scaler.fit_transform(X_raw)

    coef, _ = nnls(X, y)

    alpha = coef[:len(pks_cols)]   # PKS 部分
    beta = coef[len(pks_cols):]    # ECS 部分

    return alpha, beta, scaler, feature_cols


def apply_redeep(df, alpha, beta, scaler, feature_cols, pks_keys):
    X = scaler.transform(df[feature_cols].values.astype(float))
    n_pks = len(pks_keys)

    # score = sum(alpha * PKS) - sum(beta * ECS)
    score = X[:, :n_pks] @ alpha - X[:, n_pks:] @ beta

    df = df.copy()
    df["redeep_score"] = score
    return df


# ====================================================
# Evaluation
# ====================================================

def evaluate_with_threshold(test_df, threshold):
    # ---------- Chunk-level ----------
    y_c = test_df["hallucination_label"].values
    s_c = test_df["redeep_score"].values
    yhat_c = (s_c >= threshold).astype(int)

    chunk = {
        "AUC": roc_auc_score(y_c, s_c) if len(np.unique(y_c)) > 1 else None,
        "Accuracy": accuracy_score(y_c, yhat_c),
        "Precision": precision_score(y_c, yhat_c, zero_division=0),
        "Recall": recall_score(y_c, yhat_c, zero_division=0),
        "F1": f1_score(y_c, yhat_c, zero_division=0),
        "PCC": safe_pearsonr(s_c, y_c),
    }

    # ---------- Response-level ----------
    df = test_df.copy()
    df["pred"] = yhat_c
    df["response_group"] = df["identifier"].apply(lambda x: x.split("_span_")[0])

    grouped = df.groupby("response_group").agg({
        "redeep_score": "mean",
        "hallucination_label": "max",
        "pred": "max"
    }).reset_index()

    y_r = grouped["hallucination_label"].values
    s_r = grouped["redeep_score"].values
    yhat_r = grouped["pred"].values

    response = {
        "AUC": roc_auc_score(y_r, s_r) if len(np.unique(y_r)) > 1 else None,
        "Accuracy": accuracy_score(y_r, yhat_r),
        "Precision": precision_score(y_r, yhat_r, zero_division=0),
        "Recall": recall_score(y_r, yhat_r, zero_division=0),
        "F1": f1_score(y_r, yhat_r, zero_division=0),
        "PCC": safe_pearsonr(s_r, y_r),
    }

    return chunk, response

In [ ]:
# ====================================================
# Main
# ====================================================

INPUT_RESULT_FILE = "Llama-3.2-3B_hallurag_results.json" # 請換成 Stage 2 中的輸出檔案名字
SAVE_PATH = "redeep_nnls_eval.json"

if __name__ == "__main__":

    # ---------- Load data ----------
    df, ecs_keys, pks_keys = construct_dataframe(INPUT_RESULT_FILE)

    # 使用 dataset 內建 split
    train_df = df[df["split"] == "train"].copy()
    test_df  = df[df["split"] == "test"].copy()

    # safety：檢查 response leakage
    assert len(
        set(train_df["identifier"].str.split("_span_").str[0])
        & set(test_df["identifier"].str.split("_span_").str[0])
    ) == 0, "Response leakage detected!"

    # ---------- Feature ranking ----------
    ecs_metrics = calculate_feature_metrics(train_df, ecs_keys, "ecs")
    pks_metrics = calculate_feature_metrics(train_df, pks_keys, "pks")

    ecs_ranked = [m["key"] for m in sorted(ecs_metrics, key=lambda x: x["auc"], reverse=True)]
    pks_ranked = [m["key"] for m in sorted(pks_metrics, key=lambda x: x["auc"], reverse=True)]

    # ---------- Grid search ----------
    best_auc = -1
    best_all = None

    total_ecs = len(ecs_ranked)
    total_pks = len(pks_ranked)

    print("\n[Info] 開始 Grid Search (Top-ECS x Top-PKS)...")
    for n_ecs in range(1, total_ecs + 1):
        for n_pks in range(1, total_pks + 1):

            sel_ecs = ecs_ranked[:n_ecs]
            sel_pks = pks_ranked[:n_pks]

            alpha, beta, scaler, feature_cols = train_redeep_nnls(
                train_df, sel_pks, sel_ecs
            )

            train_scored = apply_redeep(
                train_df, alpha, beta, scaler, feature_cols, sel_pks
            )

            threshold = 0.0
            
            fpr, tpr, thresholds = roc_curve(train_scored["hallucination_label"].values, train_scored["redeep_score"].values)
            j_scores = tpr - fpr
            best_idx = np.argmax(j_scores)
            threshold = thresholds[best_idx]

            test_scored = apply_redeep(
                test_df, alpha, beta, scaler, feature_cols, sel_pks
            )

            chunk, response = evaluate_with_threshold(test_scored, threshold)

            # # 迴圈進度可視化（看現在跑到哪裡、AUC 表現）
            # print(
            #     f"n_ecs={n_ecs:2d}, n_pks={n_pks:2d} "
            #     f"-> AUC_resp={response['AUC']:.4f}, AUC_chunk={chunk['AUC']:.4f}, alpha={alpha.tolist()}, beta: {beta.tolist()}"
            # )

            if response["AUC"] is not None and response["AUC"] > best_auc:
                best_auc = response["AUC"]
                best_all = {
                    "top_ecs": sel_ecs,
                    "top_pks": sel_pks,
                    "alpha": alpha.tolist(),
                    "beta": beta.tolist(),
                    "threshold": float(threshold),
                    "chunk": chunk,
                    "response": response
                }

    print("\n================ 最終結果 (ReDeEP NNLS) ================")
    print(f"最佳 Top-ECS-head 數量: {len(best_all['top_ecs'])} → {best_all['top_ecs']}")
    print(f"最佳 Top-PKS-layer 數量: {len(best_all['top_pks'])} → {best_all['top_pks']}")
    print(f"學到的 PKS 係數 (alpha, mean): {np.mean(best_all['alpha']):.4f}")
    print(f"學到的 ECS 係數 (beta, mean):  {np.mean(best_all['beta']):.4f}")
    print(f"學到的 threshold: {best_all['threshold']:.4f}")

    table = pd.DataFrame([
        {"Level": "Chunk", **best_all["chunk"]},
        {"Level": "Response", **best_all["response"]},
    ])

    print("\n================ Evaluation Metrics ================")
    print(table.round(4))